# [실습] LangChain Tool Call

LLM은 언어 모델이지만, 특수한 형식의 출력을 통해 외부 도구와 소통할 수 있습니다.   

1. LLM에게 프롬프트를 통해 도구의 사용법을 전달합니다.   
2. LLM은 도구가 필요한 경우 파싱 가능한 출력을 생성합니다.   
3. 해당 출력을 파싱하여 실제 도구를 실행하는 과정을 자동화합니다.
4. 실행 결과를 다시 LLM에 전달하면, LLM은 결과를 참고하여 다음 출력을 생성합니다.


# 라이브러리 설정 및 LLM 불러오기

In [1]:
!pip install langchain==0.3.27 langchain_community==0.3.27 langgraph==0.6.8 langchain_google_genai langchain_openai langchain_community langchain_tavily -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 8.

In [2]:
import os
from dotenv import load_dotenv
# OPENAI_API_KEY, GOOGLE_API_KEY
load_dotenv(override=True)

True

In [3]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# Gemini: 무료 API 사용량 존재
# 안정적 서빙을 위해 분당 10개 설정
# 즉, 초당 약 0.167개 요청 (10/60)
# `https://aistudio.google.com/`에서 모델별 사용량 확인

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.167,  # 분당 10개 요청
    check_every_n_seconds=0.1,  # 100ms마다 체크
    max_bucket_size=10,  # 최대 버스트 크기
)


# LLM 초기화
try:
    llm = ChatOpenAI(model='gpt-5-mini', temperature=0.3)
    llm_nonreasoning = ChatOpenAI(model='gpt-4.1-mini', temperature=0.3)
    print("✅ GPT API 사용 가능!")
except:
    print("❌ GPT API 사용 불가- API 키를 확인하세요!")
try:
    llm_gemini = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.7,
                                rate_limiter=rate_limiter)
    llm_gemini_nonreasoning = ChatGoogleGenerativeAI(model='gemini-2.0-flash', temperature=0.3,
                                rate_limiter=rate_limiter)
    print("✅ Gemini API 사용 가능!")
except:
    print("❌ Gemini API 사용 불가- API 키를 확인하세요!")

✅ GPT API 사용 가능!
✅ Gemini API 사용 가능!


In [4]:
from langchain.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

print("필수 모듈 임포트 완료")

필수 모듈 임포트 완료


## LLM에게 어려운 질문하기   

LLM에게 학습하지 못한 범위의 질문을 전달하면 정확도가 떨어지거나 잘못된 답변을 생성합니다.   

In [5]:
llm.invoke("케이팝 데몬헌터스가 뭐야?")
# 5-mini: 24.6 까지만

AIMessage(content='어떤 걸 말씀하시는 건지 조금 더 알려주실래요? “케이팝 데몬헌터스”라는 이름은 여러 맥락에서 쓰일 수 있어서요. 예를 들어:\n\n- 유튜브/팬메이드 뮤직비디오나 웹툰·팬픽 등 팬창작물의 제목일 수 있고,  \n- 게임(모바일/인디)이나 모드의 이름일 수 있고,  \n- 혹은 특정 콘셉트의 케이팝 그룹이나 프로젝트, 팬덤 별칭일 수도 있습니다.\n\n어디에서 보셨나요(링크나 플랫폼, 이미지 등)? 아니면 그 이름으로 된 작품이나 노래에 대해 설명을 원하시나요? 원하시면 해당 맥락에 맞춰 자세히 설명해 드릴게요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 566, 'prompt_tokens': 19, 'total_tokens': 585, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E6lYuJifjywgN9sgQ2YQk4r4qH0pH', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019fab11-e12c-7c23-ba49-62e1810565e3-0', usage_metadata={'input_tokens': 19, 'output_tokens': 566, 'total_tokens': 585

In [ ]:
llm_gemini.invoke("케이팝 데몬헌터스가 뭐야?")

AIMessage(content='"케이팝 데몬헌터스(K-Pop Demon Hunters)"는 특정 작품 하나의 이름이라기보다는, **케이팝 아이돌의 화려함과 퍼포먼스에 악마나 초자연적인 존재를 사냥하는 \'데몬 헌터\'의 액션/판타지 요소를 결합한 장르나 콘셉트**를 의미합니다.\n\n즉, 무대 위에서는 빛나는 아이돌이지만, 실제로는 비밀리에 초자연적인 위협으로부터 세상을 지키는 능력을 가진 존재라는 설정이죠.\n\n주로 다음과 같은 형태로 나타납니다:\n\n1.  **웹툰/웹소설:**\n    *   가장 활발하게 볼 수 있는 형태입니다. 많은 K-Pop 기획사들이 소속 아이돌 그룹의 세계관(Universe)을 확장하기 위해 웹툰이나 웹소설을 제작하는데, 이 과정에서 아이돌 멤버들이 판타지 세계의 영웅이나 데몬 헌터로 등장하는 경우가 많습니다.\n    *   **예시:**\n        *   **ENHYPEN(엔하이픈)의 \'DARK MOON: 달의 제단\':** 뱀파이어 소년들과 늑대인간들, 그리고 특별한 능력을 가진 소녀의 이야기를 다룬 판타지 웹툰/웹소설입니다.\n        *   **TXT(투모로우바이투게더)의 \'별을 쫓는 소년들\':** 마법 능력을 가진 아이돌 멤버들이 세상을 구하는 이야기를 다룹니다.\n        *   **BTS(방탄소년단)의 \'7FATES: CHAKHO\':** 한국 전통 설화 속 범 사냥꾼(착호갑사)을 모티브로 한 도시 판타지 웹툰/웹소설입니다.\n\n2.  **게임:**\n    *   간혹 인디 게임이나 팬들이 만든 모드(Mod) 등에서 K-Pop 아이돌의 이미지를 차용하거나, 아예 K-Pop 아이돌을 모티브로 한 데몬 헌터 캐릭터가 등장하는 경우가 있습니다.\n    *   **예시:** \'Project: Bellatores\'와 같이 K-Pop 요소를 넣은 데몬 헌팅 게임 프로젝트가 개발되기도 합니다.\n\n3.  **팬픽션/AU (Alternate Universe):**\n    *   팬덤 내에

Anthropic의 기술 블로그에서는, LLM의 외연을 확장하는 방법으로 검색(Retrieval)과 도구(Tool)을 제안했습니다.   
Retrieval은 이후의 RAG에서 자세히 다루도록 하고, 이번 과정에서는 Tool에 대해 이해해 보겠습니다.

## Tool

Tool은 LLM이 이해하고 사용할 수 있는 도구입니다.   
프롬프트를 통해 설명을 전달하므로, 넓은 의미에서 Context Engineering에 해당합니다.   


In [6]:
# OpenAI API의 도구 생성 기능

from openai import pydantic_function_tool
from pydantic import BaseModel, Field

class web_search(BaseModel):
    """웹 검색을 수행합니다."""
    query: str= Field(description="""검색 키워드""")

web_search_tool = pydantic_function_tool(web_search)

web_search_tool

{'type': 'function',
 'function': {'name': 'web_search',
  'strict': True,
  'parameters': {'description': '웹 검색을 수행합니다.',
   'properties': {'query': {'description': '검색 키워드',
     'title': 'Query',
     'type': 'string'}},
   'required': ['query'],
   'title': 'web_search',
   'type': 'object',
   'additionalProperties': False},
  'description': '웹 검색을 수행합니다.'}}

위 결과를 Tool Schema라고 하며, 해당 내용이 LLM의 프롬프트에 포함되어 툴 사용 정보를 전달합니다.

## LangChain Tool 설정하기

LangChain은 Tool Calling을 쉽게 연동하기 위한 기능을 제공합니다.   


LangChain에서 자체적으로 지원하는 Tool을 사용하거나, RAG의 Retriever를 Tool로 변환하는 것도 가능합니다.     
또한, 함수를 Tool로 변환할 수도 있습니다.

가장 대표적인 built-in 툴인 Tavily Search (http://app.tavily.com/) 를 연결해 보겠습니다.

해당 도메인에 접속하여, 구글 계정으로 회원가입을 진행해 주세요.

이후, API 키를 생성하여 env 파일에 `TAVILY_API_KEY`로 추가합니다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [7]:
from langchain_tavily import TavilySearch

# 파일 업데이트 후 실행
load_dotenv(override=True)

tavily_search = TavilySearch(
    max_results=5,
    # 더 많은 옵션은 Tavily API 문서 Playground 참고
    )

In [ ]:
from rich import print as rprint
rprint(tavily_search)

TavilySearch(
    max_results=5,
    api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'), api_base_url=None)
)

In [8]:
search_docs = tavily_search.invoke("케이팝 데몬 헌터스의 의미")
search_docs

{'query': '케이팝 데몬 헌터스의 의미',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://ianpark.vc/p/kdh',
   'title': '케이팝 데몬 헌터스의 진짜 의미: 오렌지치킨과 캘리포니아롤의 시대 - by Ian Park',
   'content': '안녕하세요 이안입니다! 이번주에는 한번쯤은 들어보셨을 넷플릭스를 통해 미국을 휩쓸고 있는 K-Pop Demon Hunters(케데헌)을 보고 느낀 점을 공유드리려고 합니다.\n\n# 1. 도대체 케데헌이 뭔데?\n\n케이팝 데몬 헌터스(케데헌)는 세계적인 K-pop 걸그룹 헌트릭스(Huntr/x)가 사실은 대대로 인간계를 지켜온 악마 사냥꾼이라는 이중생활을 한다는 설정의 애니메이션 영화입니다. 이들은 노래의 힘으로 인간계와 악마계를 가르는 장벽 “혼문”을 유지하며, 라이벌 보이그룹이자 비밀리에 악마로 구성된 “사자 보이즈”의 방해에 맞서 싸웁니다. 한국계 미국인 매기 강 감독이 자신의 문화적 유산에서 영감을 받아 기획했으며, 저승사자와 탈춤 등 한국 신화를 깊이 있게 녹여낸 이 작품은 소니 픽처스 애니메이션이 제작하고 넷플릭스를 통해 전 세계에 공개되어 상업적·비평적으로 큰 성공을 거두었습니다. [...] # 이안의 주간실리콘밸리\n\n# 케이팝 데몬 헌터스의 진짜 의미: 오렌지치킨과 캘리포니아롤의 시대\n\n### 한국이냐 아니냐는 더 이상 중요하지 않습니다. 그냥 맛있으면 되는 겁니다.\n\nIan Park\n\nAug 28, 2025\n\n본 커뮤니티의 모든 내용은 대중에게 공개된 정보를 기반으로한 개인적인 의견이며 투자에 대한 조언이 아닌 전반적인 미국의 시장, VC, 스타트업, 기술 트렌드와 그에 대한 개인적인 의견들입니다. 유튜브, 링드인, 인스타그램, 스레드, 줌/커피챗신청 등 다른 다양한 채널들을 보고 싶으시다면 위의 링크를 이용하시거나 컴퓨터에서 오른쪽에 “

API 호출은 대부분 사용자가 원하는 정보보다 많은 내용을 전달합니다.   
불필요한 내용을 제거하는 함수를 구성합니다.

In [9]:
from langchain_core.tools import tool

@tool
def tavily_search(query, max_results = 5):
    """Tavily 검색을 통해 인터넷 검색 결과를 가져옵니다.
query는 검색어를 의미하며, max_results는 최대 검색 문서 수를 의미합니다.
사용자가 별도의 요청을 하지 않으면 max_results는 5를 사용하고, 최대한 검색하라고 하는 경우 20을 사용하세요."""
    tavily_search = TavilySearch(max_results=max_results)
    results = tavily_search.invoke(query)['results']

    context = ''
    for result in results:
        # result: dict
        doc_content = 'URL: '+ result.get('url') +'\n Title:'+ result.get('title') +'\n Content:' + result.get('content')
        context += doc_content + '\n\n---\n\n'

    return context

print(tavily_search.invoke({'query':'케이팝 데몬헌터스'}))

URL: https://ko.wikipedia.org/wiki/%EC%BC%80%EC%9D%B4%ED%8C%9D_%EB%8D%B0%EB%AA%AC_%ED%97%8C%ED%84%B0%EC%8A%A4
 Title:케이팝 데몬 헌터스 - 위키백과, 우리 모두의 백과사전
 Content:《케이팝 데몬 헌터스》(영어: KPop Demon Hunters)는 2025년 공개된 미국의 뮤지컬, 판타지, 코미디 애니메이션 영화이다. 소니 픽처스 애니메이션이 제작하고 넷플릭스가 배급한다. 매기 강과 크리스 아펠한스가 감독을 맡았으며, 매기 강이 구상한 이야기를 바탕으로 강, 아펠한스, 해나 맥메찬, 다냐 히메네스가 각본을 썼으며, 성우진은 아든 조, 안효섭, 메이 홍, 유지영 "유지영 (미국의 배우)"), 김윤진, 대니얼 대 킴, 켄 정, 이병헌으로 구성되어 있다. K-pop 걸 그룹 헌트릭스(HUNTR/X)가 악마 사냥꾼으로서 이중생활을 하며, 비밀리에 악령 라이벌 보이 밴드 사자 보이즈(Saja Boys)의 멤버들과 대결하는 이야기를 그린다. [...] 케이팝 데몬 헌터스는 한국 문화에 영감을 받은 이야기를 만들고자 했던 강 감독의 바람에서 비롯되었으며, 신화, 악마학, K-pop 요소를 활용하여 시각적으로 독특하고 한국의 전통문화 요소가 담긴 영화를 제작했다. 2021년 3월 소니 픽처스 애니메이션에서 제작 중인 것으로 보도되었으며, 모든 크리에이티브 팀이 참여했다. 소니 픽처스 이미지워크스에서 애니메이션화되었으며, 콘서트 조명, 사진, 뮤직 비디오의 구성은 한국 드라마와 일본의 애니메이션의 영향을 받았다. 사운드 트랙에는 마르셀루 자르부스가 작곡했으며, 여러 아티스트가 주제곡에 참여했다.

케이팝 데몬 헌터스는 2025년 6월 20일 넷플릭스에서 공개되었으며, 매우 긍정적인 평가를 받았다. [...] 뉴욕 타임스의 브랜든 유는 "케이팝 데몬 헌터스"가 "매력적이고, 재미있고, 예술적으로 강력한 독창적인 세계"라고 평가하며, "K-pop부터 K-드라마, 대량 생산되는 노래

위에서 만든 함수에, @tool decorator를 붙이면 랭체인의 툴로 변환됩니다.

## LLM에 Tool 연결하기   

생성한 툴은 llm.bind_tools()를 통해 LLM에 연결할 수 있습니다.    

In [11]:
from rich import print as rprint

In [12]:
tools = [tavily_search]

llm_with_tools = llm.bind_tools(tools)
rprint(llm_with_tools)

RunnableBinding(
    bound=ChatOpenAI(
        client=<openai.resources.chat.completions.completions.Completions object at 0x7977b74ef230>,
        async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7977b6f62fc0>,
        root_client=<openai.OpenAI object at 0x7977b7bcffe0>,
        root_async_client=<openai.AsyncOpenAI object at 0x7977b74ef290>,
        model_name='gpt-5-mini',
        model_kwargs={},
        openai_api_key=SecretStr('**********'),
        openai_proxy=None,
        stream_usage=True
    ),
    kwargs={
        'tools': [
            {
                'type': 'function',
                'function': {
                    'name': 'tavily_search',
                    'description': 'Tavily 검색을 통해 인터넷 검색 결과를 가져옵니다.\nquery는 검색어를 의미하며, 
max_results는 최대 검색 문서 수를 의미합니다.\n사용자가 별도의 요청을 하지 않으면 max_results는 5를 사용하고, 
최대한 검색하라고 하는 경우 20을 사용하세요.',
                    'parameters': {
                        'properties': {'query': {}, 'max_results': {'default': 5}},
                        'required': ['query'],
                        'type': 'object'
                    }
                }
            }
        ]
    },
    config={},
    config_factories=[]
)

랭체인에서, tool 정보는 tools에 저장되는데요.   
해당 내용은 랭체인 내부에서 json Schema 형식으로 프롬프트에 포함됩니다.

In [13]:
llm_with_tools.kwargs['tools']

[{'type': 'function',
  'function': {'name': 'tavily_search',
   'description': 'Tavily 검색을 통해 인터넷 검색 결과를 가져옵니다.\nquery는 검색어를 의미하며, max_results는 최대 검색 문서 수를 의미합니다.\n사용자가 별도의 요청을 하지 않으면 max_results는 5를 사용하고, 최대한 검색하라고 하는 경우 20을 사용하세요.',
   'parameters': {'properties': {'query': {}, 'max_results': {'default': 5}},
    'required': ['query'],
    'type': 'object'}}}]

LLM은 프롬프트로 주어지는 툴 정보를 바탕으로 Tool의 사용을 결정합니다.    
Schema를 통해, 툴의 의미와 사용 방법, 형식을 이해합니다.

Tool을 탑재한 LLM을 실행해 보겠습니다.

In [14]:
from langchain_core.output_parsers import StrOutputParser

In [15]:
prompt = ChatPromptTemplate(
    [
        ('system', '주어진 검색 툴을 사용해서 질문에 답변하세요.'),
        ('human', '{query}')
    ]
)
# parser = StrOutputParser()
chain = prompt | llm_with_tools
# chain = prompt | llm | parser
# tool_call_msg = chain.invoke('삼성SDS의 GPUaaS 서비스가 뭐야?')
tool_call_msg = llm_with_tools.invoke('삼성SDS의 GPUaaS 서비스가 뭐야?')

rprint(tool_call_msg)

AIMessage(
    content='',
    additional_kwargs={
        'tool_calls': [
            {
                'id': 'call_iVSVRL90Ra3q0UqM2xmdPvua',
                'function': {
                    'arguments': '{"query":"삼성SDS GPUaaS 서비스", "max_results":5}',
                    'name': 'tavily_search'
                },
                'type': 'function'
            }
        ],
        'refusal': None
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 102,
            'prompt_tokens': 208,
            'total_tokens': 310,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 64,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
        },
        'model_name': 'gpt-5-mini-2025-08-07',
        'system_fingerprint': None,
        'id': 'chatcmpl-E6lcLFBnaupMIPRvXh3rGBboNNV6i',
        'service_tier': 'default',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='run--019fab15-22e3-7b01-a5c5-83d2c4d8cdb1-0',
    tool_calls=[
        {
            'name': 'tavily_search',
            'args': {'query': '삼성SDS GPUaaS 서비스', 'max_results': 5},
            'id': 'call_iVSVRL90Ra3q0UqM2xmdPvua',
            'type': 'tool_call'
        }
    ],
    usage_metadata={
        'input_tokens': 208,
        'output_tokens': 102,
        'total_tokens': 310,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 64}
    }
)

In [16]:
tool_call_msg

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_iVSVRL90Ra3q0UqM2xmdPvua', 'function': {'arguments': '{"query":"삼성SDS GPUaaS 서비스", "max_results":5}', 'name': 'tavily_search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 208, 'total_tokens': 310, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E6lcLFBnaupMIPRvXh3rGBboNNV6i', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019fab15-22e3-7b01-a5c5-83d2c4d8cdb1-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': '삼성SDS GPUaaS 서비스', 'max_results': 5}, 'id': 'call_iVSVRL90Ra3q0UqM2xmdPvua', 'type': 'tool_call'}], usage_metadata={'input_to

In [17]:
tool_list = {'tavily_search': tavily_search}
tool_name = tool_call_msg.tool_calls[0]['name']
tool_exec = tool_list[tool_name]
tool_msg = tool_exec.invoke(tool_call_msg.tool_calls[0])

messages = []
messages.append(HumanMessage('삼성SDS의 GPUaaS 서비스가 뭐야?'))
messages.append(tool_call_msg)
messages.append(tool_msg)
chain = llm_with_tools | StrOutputParser()
result = chain.invoke(messages)

In [ ]:
print(result)

간단히 말하면, 삼성SDS의 GPUaaS(GPU as a Service)는 고성능 GPU 자원을 클라우드 형태로 빌려 쓰는 서비스입니다. 기업이나 개발자가 비싼 GPU를 직접 구매·운영하지 않아도, 필요할 때 바로 고성능 연산 환경을 쓸 수 있게 해주며 AI 모델 실험·학습·배포 전 과정에 맞춘 환경과 운영 지원을 제공합니다.

주요 내용(요약)
- 무엇을 제공하나: 엔비디아 H100·B300(Blackwell 계열) 등 고성능 GPU 기반 인프라를 온디맨드 또는 구독형으로 제공. 베어메탈 GPU 클러스터도 지원.
- 대상 작업: AI 모델 실험/개발, 대규모 분산 학습(대형 LLM 학습), 추론(서빙), 고성능 컴퓨팅(HPC) 등.
- 장점: 초기 투자 비용 절감(구매 불필요), 즉시 사용 가능한 환경, 대규모 분산 처리에 최적화된 구성, MLOps·AI 서비스와 연계, 24/7 장애 대응 및 전문 기술 지원, 기업용 보안·운영 체계.
- 과금 방식: 구독형(또는 사용량 기반)으로 필요한 만큼만 비용 지불(자세한 요금은 상담 필요).
- 추가 기능/계획: 서버리스 추론, 자동 분산 학습 같은 서비스 연계 기능을 제공하거나 계획 중이라는 안내가 있음.

누가 쓰면 좋은가
- AI 모델을 개발·배포하는 기업(스타트업 ~ 대기업)
- 대규모 학습 자원이 필요한 연구팀
- 자체 인프라 관리 부담을 줄이고 싶은 개발팀

원하면 삼성SDS 공식 페이지(서비스 설명)와 현재 제공되는 GPU 종류/사양, 요금제, 도입 절차를 찾아 자세히 보여드리거나, 귀하의 사용 사례에 맞는 추천 구성을 함께 제안해 드릴게요. 어떤 정보가 필요하세요?


In [20]:
prompt = ChatPromptTemplate(
    [
        ('system', '주어진 검색 툴을 사용해서 질문에 답변하세요.'),
        ('human', '{query}'),
        ('ai', '{tool_call_msg}'),
        ('ai', '{tool_msg}')
    ]
)
parser = StrOutputParser()
chain = prompt | llm_with_tools | parser

result = chain.invoke({'query': '삼성SDS의 GPUaaS 서비스가 뭐야?', 'tool_call_msg' : tool_call_msg, 'tool_msg' : tool_msg})

In [21]:
print(result)

간단히 말하면 GPUaaS(GPU as a Service)는 클라우드에서 GPU 연산 자원을 시간/구독 단위로 빌려 쓰는 서비스입니다. 직접 GPU 서버를 사서 관리할 필요 없이 AI 학습·추론, 고성능 연산(HPC), 그래픽 렌더링 등 GPU가 필요한 워크로드를 바로 돌릴 수 있게 해줍니다.

삼성SDS GPUaaS(요약)
- 플랫폼: 삼성 클라우드 플랫폼(SCP) 기반의 GPU 구독/임대 서비스. 즉시 사용 가능한 GPU 인스턴스를 제공합니다.
- 제공 특징:
  - 고성능 GPU 인프라(공식 페이지 기준 “즉시 사용 가능한 고성능 GPU”)
  - 엔비디아 최신 GPU 라인업을 제공(공개 자료에서 A100, H100, L40S, 최근에는 B300 기반 인프라 언급)
  - 도입 상담부터 구축·운영까지 컨설팅·기술지원 제공(전문 기술지원/운영 안정성 강조)
  - 국내 대규모 GPU 보유·운영 역량을 내세움(대형 고객·연구용에 적합)
- 타깃: 기업·연구기관 등 대규모 AI 학습이나 추론, HPC 워크로드를 가진 조직

참고자료(자세히 보려면)
- 삼성SDS GPUaaS 소개: https://www.samsungsds.com/kr/gpuaas/gpuaas.html
- 이벤트/설명 페이지(제공 GPU·장점 설명): https://www.samsungsds.com/kr/event/gpuaas-event1.html
- 관련 기사·시장망: DidimGPU / Vessl 등(국내 도입 GPU 정보·시장 현황)

원하시면
- 예상 비용 구조(온디맨드 vs 구독)나 성능 비교(A100 vs H100 vs B300 등)로 어떻게 선택할지 정리해 드리거나,
- 귀사(또는 사용 목적)에 맞는 인스턴스 추천을 해드릴게요. 어떤 용도로 쓰실 계획인가요?


`tool_calls`에 포함된 name을 활용하면 툴 결과를 전달할 수 있습니다.   
`name` 값은 문자열이므로, dictionary를 통해 연결합니다.

In [ ]:
tool_call_msg.tool_calls

[{'name': 'tavily_search',
  'args': {'query': '삼성SDS GPUaaS 서비스', 'max_results': 5},
  'id': 'call_3Wm6eG64P7lhGPPT1oeigABQ',
  'type': 'tool_call'}]

Tool에 tool_call을 입력해 invoke를 수행합니다.

In [22]:
tool_list = {'tavily_search': tavily_search}
tool_exec = tool_list[tool_call_msg.tool_calls[0]['name']]
tool_exec

StructuredTool(name='tavily_search', description='Tavily 검색을 통해 인터넷 검색 결과를 가져옵니다.\nquery는 검색어를 의미하며, max_results는 최대 검색 문서 수를 의미합니다.\n사용자가 별도의 요청을 하지 않으면 max_results는 5를 사용하고, 최대한 검색하라고 하는 경우 20을 사용하세요.', args_schema=<class 'langchain_core.utils.pydantic.tavily_search'>, func=<function tavily_search at 0x7977b697e160>)

In [23]:
tool_msg = tool_exec.invoke(tool_call_msg.tool_calls[0])
print(tool_msg.content)

URL: https://www.samsungsds.com/kr/gpuaas/gpuaas.html
 Title:삼성 클라우드 플랫폼 GPUaaS | 즉시 사용 가능한 고성능 GPU 도입
 Content:GPU 리소스를 할당 받아 사용하는 서비스입니다. 삼성SDS의 B300 GPUaaS는 B300 기반 인프라, 즉시 사용 가능한 고성능 GPU, 전문 기술 지원, 안정적인 인프라 운영,

---

URL: https://www.samsungsds.com/kr/event/gpuaas-event1.html
 Title:GPU as a Service GPUaaS 고민, 삼성SDS가 모두 해결해 드립니다 | 이벤트
 Content:삼성SDS GPUaaS가 좋은 이유 · 1. NVIDIA H100 등 국내 최대 GPU 보유량 기반, 국내 최대 규모 GPU 서비스 운영 · 2. 도입 상담부터 구축, 원활한 운영까지 컨설턴트의 End

---

URL: https://vessl.ai/ko/blog/gpuaas-vessl
 Title:'GPU' 빌려 쓰는 시대, '이 기업'들 주목도↑
 Content:VESSL AI
South Korea

# 'GPU' 빌려 쓰는 시대, '이 기업'들 주목도↑

VESSL AI
'GPU' 빌려 쓰는 시대, '이 기업'들 주목도↑

GPUaaS 시장은 2033년 144억5,840만달러(약 21조4,000억원)에 이를 전망이에요. 국내에서는 삼성SDS가 1월에 SCP(삼성 클라우드 플랫폼) 기반 GPU 구독 서비스를 출시하면서 엔비디아 B300을 국내 최초로 도입했고, 지난해 4분기 GPUaaS가 포함된 CSP 사업 매출이 2,713억원으로 전년 대비 17% 늘었어요. 베슬AI는 자체 개발한 GPUaaS '네오클라우드(Neocloud)'를 기반으로 미국·이스라엘·핀란드·일본 등 8개국 데이터센터와 계약하고, 스탠포드·버클리 등 글로벌 100여 기업·연구기관에 서비스를 공급하고 있어요.

원문 보기 →

VESSL AI

VESSL AI

### 뉴스

ToolMessage라는 새로운 형태의 메시지가 생성되었습니다.

HumanMessage, AIMessage(Tool Call), ToolMessage를 모두 전달합니다.

In [24]:
tool_msg

ToolMessage(content="URL: https://www.samsungsds.com/kr/gpuaas/gpuaas.html\n Title:삼성 클라우드 플랫폼 GPUaaS | 즉시 사용 가능한 고성능 GPU 도입\n Content:GPU 리소스를 할당 받아 사용하는 서비스입니다. 삼성SDS의 B300 GPUaaS는 B300 기반 인프라, 즉시 사용 가능한 고성능 GPU, 전문 기술 지원, 안정적인 인프라 운영,\n\n---\n\nURL: https://www.samsungsds.com/kr/event/gpuaas-event1.html\n Title:GPU as a Service GPUaaS 고민, 삼성SDS가 모두 해결해 드립니다 | 이벤트\n Content:삼성SDS GPUaaS가 좋은 이유 · 1. NVIDIA H100 등 국내 최대 GPU 보유량 기반, 국내 최대 규모 GPU 서비스 운영 · 2. 도입 상담부터 구축, 원활한 운영까지 컨설턴트의 End\n\n---\n\nURL: https://vessl.ai/ko/blog/gpuaas-vessl\n Title:'GPU' 빌려 쓰는 시대, '이 기업'들 주목도↑\n Content:VESSL AI\nSouth Korea\n\n# 'GPU' 빌려 쓰는 시대, '이 기업'들 주목도↑\n\nVESSL AI\n'GPU' 빌려 쓰는 시대, '이 기업'들 주목도↑\n\nGPUaaS 시장은 2033년 144억5,840만달러(약 21조4,000억원)에 이를 전망이에요. 국내에서는 삼성SDS가 1월에 SCP(삼성 클라우드 플랫폼) 기반 GPU 구독 서비스를 출시하면서 엔비디아 B300을 국내 최초로 도입했고, 지난해 4분기 GPUaaS가 포함된 CSP 사업 매출이 2,713억원으로 전년 대비 17% 늘었어요. 베슬AI는 자체 개발한 GPUaaS '네오클라우드(Neocloud)'를 기반으로 미국·이스라엘·핀란드·일본 등 8개국 데이터센터와 계약하고, 스탠포드·버클리 등 글로벌 100여 기업·연구기관에 서비스를 공급하

In [25]:
messages= []

messages.append(HumanMessage('삼성SDS의 GPUaaS 서비스가 뭐야?'))
# 질문이 들어있는 HumanMessage + 툴 요청이 들어있는 AIMessage + 툴 결과가 들어있는 ToolMessage
messages.append(tool_call_msg)
messages.append(tool_msg)

`Query-Tool Call-Tool`의 형식은 가장 기본적인 툴 사용 방법입니다.

In [26]:
result = llm_with_tools.invoke(messages)
print(result.content)

간단히 정리하면 다음과 같습니다.

1) GPUaaS(GPU as a Service)란?
- 물리적 GPU 하드웨어를 직접 사지 않고, 클라우드 형태로 GPU 연산자원(가속기)을 임대·사용하는 서비스입니다.
- 필요할 때 즉시 GPU를 할당받아 머신러닝 학습·추론, 모델 개발·테스트, 고성능 연산(HPC), 그래픽 렌더링 등 GPU가 필요한 작업을 처리할 수 있습니다.
- 장점: 초기 투자 절감, 필요한 만큼 확장/축소, 관리형 인프라(운영·패치·모니터링 등) 제공.

2) 삼성SDS의 GPUaaS(요약)
- 삼성SDS는 자사 클라우드 플랫폼(SCP)을 통해 GPUaaS를 제공하며 “즉시 사용 가능한 고성능 GPU”, “전문 기술 지원”, “안정적 인프라 운영”을 강조합니다.
- 제품·홍보자료에 따르면 NVIDIA H100/A100/L40S 등 최신 GPU 계열을 보유·운영하고 있으며(특히 B300 기반 인프라 도입 사례 언급), 국내에서 대규모 GPU 풀을 운영하는 것을 강점으로 내세웁니다.
- 기업 대상 컨설팅·도입 지원, 운영·보안 관리, SLA(가용성) 제공 등 엔터프라이즈형 서비스 성격이 강합니다.

3) 누가 쓰면 좋은가?
- 대규모 AI 모델 학습이 필요한 기업·연구소
- 실시간 추론을 위한 GPU 인프라가 필요한 서비스
- 자체 인프라 운영 부담을 줄이고 싶은 기업(관리형 서비스 선호)
- PoC/프로토타입 단계에서 초기 투자 없이 GPU 성능을 검증하려는 팀

4) 고려할 점
- 필요한 GPU 종류(H100, A100 등)와 메모리/연산 요구량에 따라 비용과 성능 차이 발생
- 요금체계(시간당/구독형/인스턴스 유형), 네트워크(저지연 요구 시), 데이터 보안·컴플라이언스 요구사항 확인 필요
- SLA, 기술지원 범위(컨설팅 포함 여부), 온프레미스 연동 가능성 등도 따져야 함

5) 시작 방법
- 삼성SDS GPUaaS 공식 페이지나 영업에 문의해 상세 사양·요금·시험 사용(Trial) 가능 여부 상담 받기
- 사용 목적(학습 vs 추론), 모

In [ ]:
result = chain.invoke(messages)
print(result.content)

간단히 말하면, 삼성SDS의 GPUaaS는 “GPU를 서비스 형태로 제공”하는 클라우드 상품으로, 복잡한 인프라 관리 없이 고성능 GPU 자원을 바로 쓰면서 AI 모델 개발·학습·추론을 빠르게 진행할 수 있게 해주는 서비스입니다.

주요 내용(요약)
- 서비스 개념: 클라우드 기반 GPU 리소스(임대형) + 운영·지원 서비스를 묶은 형태. 인프라 설치·관리 없이 GPU 연산을 즉시 사용 가능.
- 대상 워크로드: 머신러닝/딥러닝 학습 및 실험, 대규모 모델(LLM) 학습, 추론 서빙, 3D 렌더링, 클라우드 게임 등 GPU 연산이 필요한 작업.
- 제공 인프라: NVIDIA H100 등 최신 고성능 GPU 제공(삼성SDS는 국내 최대 수준의 GPU 보유·운영을 내세움). 베어메탈 GPU 클러스터(대규모 학습용) 등 옵션도 있음.
- 운영·지원: 도입 상담 → 구축 → 운영까지 컨설팅·기술지원 제공. 24시간 장애 대응 및 운영 노하우 기반 관리.
- 제품/포지셔닝: 삼성 클라우드 플랫폼 기반의 GPUaaS / B300 GPUaaS 등(즉시 사용 가능한 고성능 GPU, MLOps·AI 서비스 연계 강조).
- 장점: 초기 투자·인프라 복잡성 감소, 필요할 때만 확장·사용 가능, 전문 운영 지원으로 안정성 확보, 기업용(온프레 대비 비용/유연성 이점).
- 차별점(삼성SDS 주장): 국내 최대 규모 GPU 운용·보유, 신속한 기술 지원, 엔드투엔드 컨설팅.

실제 사용 흐름(예시)
1. 상담으로 워크로드·성능 요구 파악
2. 적합한 GPU 유형(예: H100 기반 베어메탈/가상머신 등)과 요금 플랜 결정
3. 계정/리소스 할당 후 즉시 개발·학습 시작
4. 필요 시 스케일업·스케일아웃, 운영/모니터링/장애대응 지원

원하시면 더 자세히 알려드릴게요:
- 가격·과금 방식(시간당/구독 등) 조회
- 제공되는 GPU 종류/사양과 선택 가이드
- SLA(가용성), 보안·네트워크 옵션
- 삼성SDS 공식 페이지 링크 및 연락처

어떤 정보가 더 필요하세요? (예: 가격 비

In [ ]:
result = llm.invoke(messages)
print(result.content)

간단히 말하면, 삼성SDS의 GPUaaS(GPU as a Service)는 기업들이 직접 GPU 인프라를 사서 운영하지 않아도 클라우드 형태로 고성능 GPU를 즉시 사용할 수 있게 해 주는 서비스입니다. AI 모델 학습·추론, 대규모 분산 학습, 3D 렌더링 등 GPU 연산이 필요한 워크로드를 대상으로 합니다.

주요 특징
- 고성능 GPU 제공: NVIDIA H100 등 최신 GPU를 포함해 대규모 GPU 풀을 보유(삼성SDS 발표 기준).  
- 다양한 제공 형태: 베어메탈 GPU 클러스터(대규모 학습용) 등 목적에 맞는 인스턴스/구성 선택 가능.  
- 즉시 사용·관리 편의성: 복잡한 인프라 구축 없이 바로 인스턴스 할당·사용 가능.  
- 운영·지원: 삼성SDS의 운영 노하우와 24/7 장애 대응, 컨설팅·기술지원 제공.  
- 연계 서비스: MLOps, 데이터플랫폼, 클라우드 서비스와 연계해 개발→배포까지 지원.

주요 사용 사례
- 대형 AI 모델(LLM) 학습 및 분산 훈련  
- 추론(서빙)·실시간 응답 서비스  
- 모델 실험(프로토타이핑), 하이퍼파라미터 탐색  
- 그래픽 렌더링·시뮬레이션 등 고성능 컴퓨팅

장점
- 초기 투자·운영 부담 감소(하드웨어 구매·관리 불필요)  
- 필요할 때만 확장·축소 가능해 비용 효율성 확보  
- 전문 지원으로 도입 속도·안정성 향상

주의할 점 / 고려사항
- 비용 구조 확인(시간 단위/구독형/견적 방식 등)  
- 데이터 전송량·보안(민감 데이터의 온프레미스 요건 여부)  
- GPU 리소스 가용성(대규모 예약 시 사전 협의 필요)

시작 방법
1. 사용 목적(학습/추론/렌더링), 성능 요구량(NVIDIA H100 등), 예산을 정리.  
2. 삼성SDS GPUaaS 상담/견적 요청 또는 데모 신청.  
3. 설정·테스트 후 운영·확장.

더 자세한 정보나 견적이 필요하면 삼성SDS 공식 GPUaaS 소개 페이지를 보거나(삼성 클라우드 플랫폼 GPUaaS), 삼성SDS에 직접 문의해 보세요. 필요하면